In [3]:
from pathlib import Path
import re

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.plot import plotting_extent
from shapely.geometry import box

# --- Configuration ---
GMW_DIR = Path(r"D:\MANGROVES\GMW\V4\gmw_mng_ext_v4112_stack\reprojected_54034")
OUT_DIR = Path(r"D:\MANGROVES\Research_Article_Clean\Created_Data\Loss_Age_2025_Rasters")
SHP_VAL_PATH = Path(
    r"D:\MANGROVES\Research_Article_Clean\Created_Data\gmwv4_mask_anthropogenic_validation.shp"
)

MASK_FIELD = "mask_artif"
PRISTINE_MASK_VALUE = 0
REF_YEAR = 2025
MANGROVE_2025_BAND = 41
START_YEAR = None  # None -> calculé automatiquement : REF_YEAR - (MANGROVE_2025_BAND - 1)
OUT_NODATA = 0
OVERWRITE = True
PLOT_RESULTS = False
ZOOM_PAD_PX = 25
LOG_EVERY = 20
EXPLICIT_TILES = None  # Ex. ["N26E066"] pour tester une seule tuile, ou None pour toutes

OUT_DIR.mkdir(parents=True, exist_ok=True)
start_year = START_YEAR if START_YEAR is not None else REF_YEAR - (MANGROVE_2025_BAND - 1)
allowed_ids = {t.strip().upper() for t in EXPLICIT_TILES if t.strip()} if EXPLICIT_TILES else None
tile_pattern = re.compile(r"([NS]\d{2}[EW]\d{3})", re.IGNORECASE)

# --- 1. Chargement des géométries 'pristine' (mask_artif == 0) ---
if not SHP_VAL_PATH.exists():
    raise FileNotFoundError(f"Shapefile introuvable : {SHP_VAL_PATH}")

gdf = gpd.read_file(SHP_VAL_PATH)
if MASK_FIELD not in gdf.columns:
    raise KeyError(f"Champ '{MASK_FIELD}' absent. Colonnes disponibles : {list(gdf.columns)}")

pristine_gdf = gdf[gdf[MASK_FIELD] == PRISTINE_MASK_VALUE].copy()
pristine_gdf = pristine_gdf[pristine_gdf.geometry.notna() & (~pristine_gdf.geometry.is_empty)]
del gdf

# --- 2. Découverte des tuiles GMW 'pristine' ---
if not GMW_DIR.exists():
    raise FileNotFoundError(f"Dossier GMW introuvable : {GMW_DIR}")

pristine_tiles = []
for tif_path in sorted(GMW_DIR.glob("*.tif")):
    match = tile_pattern.search(tif_path.stem)
    tile_id = match.group(1).upper() if match else tif_path.stem

    if allowed_ids is not None and tile_id not in allowed_ids:
        continue

    if pristine_gdf.empty:
        continue

    with rasterio.open(tif_path) as src:
        bounds = src.bounds
        raster_crs = src.crs

    tile_poly = box(bounds.left, bounds.bottom, bounds.right, bounds.top)
    tile_gdf = gpd.GeoDataFrame({"geometry": [tile_poly]}, crs=raster_crs)
    if pristine_gdf.crs is not None and tile_gdf.crs is not None and pristine_gdf.crs != tile_gdf.crs:
        tile_gdf = tile_gdf.to_crs(pristine_gdf.crs)

    if pristine_gdf.geometry.intersects(tile_gdf.geometry.iloc[0]).any():
        pristine_tiles.append((tile_id, tif_path))

total_tiles = len(pristine_tiles)
print(f"Tuiles 'pristine' détectées ({MASK_FIELD} == {PRISTINE_MASK_VALUE}) : {total_tiles}")

# --- 3. Traitement linéaire par tuile (en mémoire, sans fichiers intermédiaires) ---
succeeded = []
failed = []

for idx, (tile_id, stack_path) in enumerate(pristine_tiles, start=1):
    out_path = OUT_DIR / f"loss_age_{REF_YEAR}_{tile_id}.tif"

    if out_path.exists() and not OVERWRITE:
        succeeded.append(out_path)
        if idx % LOG_EVERY == 0 or idx == total_tiles:
            print(f"[{idx}/{total_tiles}] tuiles traitées ({len(succeeded)} ok, {len(failed)} échec(s))")
        continue

    try:
        with rasterio.open(stack_path) as src:
            profile = src.profile.copy()
            nodata = src.nodata
            n_bands = src.count
            shape = (src.height, src.width)
            transform = src.transform
            extent = plotting_extent(src)

            if n_bands < MANGROVE_2025_BAND:
                raise ValueError(
                    f"Attendu au moins {MANGROVE_2025_BAND} bandes dans {stack_path.name}, trouvé {n_bands}."
                )

            band_to_year = {b: start_year + (b - 1) for b in range(1, n_bands + 1)}

            presence_sum = np.zeros(shape, dtype=np.uint16)
            valid_any = np.zeros(shape, dtype=bool)
            pre_2025_bands = {}
            arr_2025 = None

            for b in range(1, n_bands + 1):
                arr = src.read(b)
                if np.issubdtype(arr.dtype, np.floating):
                    valid = np.isfinite(arr)
                    if nodata is not None:
                        valid &= ~np.isnan(arr) if np.isnan(nodata) else (arr != nodata)
                else:
                    valid = np.ones(shape, dtype=bool)
                    if nodata is not None:
                        valid &= arr != nodata

                presence_sum += (valid & (arr == 1)).astype(np.uint16)
                valid_any |= valid

                if b < MANGROVE_2025_BAND:
                    pre_2025_bands[b] = arr
                elif b == MANGROVE_2025_BAND:
                    arr_2025 = arr

        if arr_2025 is None:
            raise RuntimeError(f"Lecture impossible de la bande {MANGROVE_2025_BAND} sur {stack_path.name}")

        # ever_mangrove
        ever_mask = valid_any & (presence_sum >= 1)
        del presence_sum, valid_any

        # Inversion bande 41 (2025) en mémoire : nodata -> 1 ; présent/valide -> exclu
        if np.issubdtype(arr_2025.dtype, np.floating):
            valid_2025 = np.isfinite(arr_2025)
            if nodata is not None:
                valid_2025 &= ~np.isnan(arr_2025) if np.isnan(nodata) else (arr_2025 != nodata)
        else:
            valid_2025 = np.ones(shape, dtype=bool)
            if nodata is not None:
                valid_2025 &= arr_2025 != nodata

        inv_2025_is_one = ~valid_2025
        del arr_2025, valid_2025

        # Pixels candidats : ever_mangrove == 1 ET bande 41 inversée == 1
        candidates = ever_mask & inv_2025_is_one
        del ever_mask, inv_2025_is_one

        # Recherche de la dernière année de présence (== 1) avant REF_YEAR
        last_year_arr = np.zeros(shape, dtype=np.uint16)
        unassigned = candidates.copy()
        del candidates

        for b in range(MANGROVE_2025_BAND - 1, 0, -1):
            yr = band_to_year[b]
            if yr >= REF_YEAR or b not in pre_2025_bands:
                continue
            arr = pre_2025_bands.pop(b)
            if np.issubdtype(arr.dtype, np.floating):
                valid_b = np.isfinite(arr)
                if nodata is not None:
                    valid_b &= ~np.isnan(arr) if np.isnan(nodata) else (arr != nodata)
            else:
                valid_b = np.ones(shape, dtype=bool)
                if nodata is not None:
                    valid_b &= arr != nodata

            matched = unassigned & valid_b & (arr == 1)
            if np.any(matched):
                last_year_arr[matched] = np.uint16(yr)
                unassigned &= ~matched
            if not np.any(unassigned):
                break

        pre_2025_bands.clear()
        del unassigned

        # Conversion en âge par rapport à REF_YEAR (2025)
        valid_age = (last_year_arr > 0) & (last_year_arr <= REF_YEAR)
        age_arr = np.zeros(shape, dtype=np.uint8)
        age_arr[valid_age] = (REF_YEAR - last_year_arr[valid_age]).astype(np.uint8)
        del last_year_arr

        # Écriture du raster final uniquement
        out_profile = profile.copy()
        out_profile.update({
            "driver": "GTiff",
            "dtype": "uint8",
            "count": 1,
            "nodata": OUT_NODATA,
            "compress": "lzw",
        })
        with rasterio.open(out_path, "w", **out_profile) as dst:
            dst.write(age_arr, 1)

        dated_count = int(np.sum(valid_age))
        succeeded.append(out_path)

        # Affichage carte optionnel (uniquement si PLOT_RESULTS = True)
        if PLOT_RESULTS and dated_count > 0:
            ages = np.unique(age_arr[valid_age])
            fig, axes = plt.subplots(1, 2, figsize=(14, 6))

            masked_full = np.ma.masked_where(~valid_age, age_arr)
            axes[0].imshow(masked_full, cmap="magma_r", extent=extent, origin="upper")
            axes[0].set_facecolor("#f1f5f9")
            axes[0].set_title(f"Tuile complète — Âge (réf. {REF_YEAR}) — {tile_id}", fontsize=11, fontweight="bold")
            axes[0].set_xlabel("X (m)")
            axes[0].set_ylabel("Y (m)")

            rows, cols = np.where(valid_age)
            r0 = max(0, int(rows.min()) - ZOOM_PAD_PX)
            r1 = min(age_arr.shape[0], int(rows.max()) + ZOOM_PAD_PX + 1)
            c0 = max(0, int(cols.min()) - ZOOM_PAD_PX)
            c1 = min(age_arr.shape[1], int(cols.max()) + ZOOM_PAD_PX + 1)

            sub_masked = np.ma.masked_where(~valid_age[r0:r1, c0:c1], age_arr[r0:r1, c0:c1])
            xmin, ymax = transform * (c0, r0)
            xmax, ymin = transform * (c1, r1)

            im1 = axes[1].imshow(
                sub_masked,
                cmap="magma_r",
                vmin=int(ages.min()),
                vmax=int(ages.max()),
                extent=(xmin, xmax, ymin, ymax),
                origin="upper",
            )
            axes[1].set_facecolor("#f1f5f9")
            axes[1].set_title(f"Zoom — Âge (réf. {REF_YEAR}) — {tile_id}", fontsize=11, fontweight="bold")
            axes[1].set_xlabel("X (m)")
            axes[1].set_ylabel("Y (m)")

            cbar = fig.colorbar(im1, ax=axes.ravel().tolist(), fraction=0.03, pad=0.03)
            cbar.set_label(f"Âge par rapport à {REF_YEAR} (années)")
            if len(ages) <= 15:
                cbar.set_ticks(ages)
                cbar.set_ticklabels([str(int(a)) for a in ages])

            plt.tight_layout()
            plt.show()

    except Exception as exc:
        print(f"  -> Erreur sur {tile_id} : {exc}")
        failed.append((tile_id, str(exc)))

    if idx % LOG_EVERY == 0 or idx == total_tiles:
        print(f"[{idx}/{total_tiles}] tuiles traitées ({len(succeeded)} ok, {len(failed)} échec(s))")

print(f"\nTerminé : {len(succeeded)} réussie(s), {len(failed)} échec(s) -> {OUT_DIR}")

Tuiles 'pristine' détectées (mask_artif == 0) : 1466


C:\Users\walcker\AppData\Local\Temp\ipykernel_10320\2351598611.py:112: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  arr = src.read(b)


[20/1466] tuiles traitées (20 ok, 0 échec(s))
[40/1466] tuiles traitées (40 ok, 0 échec(s))
[60/1466] tuiles traitées (60 ok, 0 échec(s))
[80/1466] tuiles traitées (80 ok, 0 échec(s))
[100/1466] tuiles traitées (100 ok, 0 échec(s))
[120/1466] tuiles traitées (120 ok, 0 échec(s))
[140/1466] tuiles traitées (140 ok, 0 échec(s))
[160/1466] tuiles traitées (160 ok, 0 échec(s))
[180/1466] tuiles traitées (180 ok, 0 échec(s))
[200/1466] tuiles traitées (200 ok, 0 échec(s))
[220/1466] tuiles traitées (220 ok, 0 échec(s))
[240/1466] tuiles traitées (240 ok, 0 échec(s))
[260/1466] tuiles traitées (260 ok, 0 échec(s))
[280/1466] tuiles traitées (280 ok, 0 échec(s))
[300/1466] tuiles traitées (300 ok, 0 échec(s))
[320/1466] tuiles traitées (320 ok, 0 échec(s))
[340/1466] tuiles traitées (340 ok, 0 échec(s))
[360/1466] tuiles traitées (360 ok, 0 échec(s))
[380/1466] tuiles traitées (380 ok, 0 échec(s))
[400/1466] tuiles traitées (400 ok, 0 échec(s))
[420/1466] tuiles traitées (420 ok, 0 échec(s))
